In [ ]:
import torch
print(torch.device("cuda"))

In [ ]:
import dataset as ds
dt = ds.Coco("/home/wanderer2414/coco2017/")
x = dt.getTrainTensor(0)
ds.ImgWrite("test0.png", ds.TensorToImg(x))

In [ ]:
# import torch
# x = dt.getTrainTensor(0)
# avg = torch.nn.AvgPool2d(3, 1, 1)
# sq: torch.Tensor = x.square()
# sum = avg(x)*9
# sum2 = avg(sq)*9
# sep: torch.Tensor = 9*sq + sum2 - 2*x*sum
# sep = sep.sum(dim=1, keepdim=True)
# sep = torch.max_pool2d(sep, 3, 1, 1)
# sep = 1-sep/sep.max()
# #
# Gx = torch.tensor(
#     [[-1, 0, 1],
#      [-2, 0, 2],
#      [-1, 0, 1]], dtype=torch.float32
# ).reshape(1, 1, 3, 3)

# Gy = torch.tensor(
#     [[-1, -2, -1],
#      [ 0,  0,  0],
#      [ 1,  2,  1]], dtype=torch.float32
# ).reshape(1, 1, 3, 3)

# Gx = Gx.repeat(3, 1, 1, 1)
# Gy = Gy.repeat(3, 1, 1, 1)
# B, C, H, W = x.shape
# print(x.shape, Gx.shape)
# gx = torch.nn.functional.conv2d(x, Gx, padding=1, groups=3)
# gy = torch.nn.functional.conv2d(x, Gy, padding=1, groups=3)
# x = torch.nn.functional.avg_pool2d(x, 20, 20)
# x = torch.nn.functional.interpolate(x, size=(H, W), mode="nearest")
# x = (gx * gx + gy * gy).sqrt()
# x -= x.min()
# x = 1 - x/x.max()
# x = x.sum(dim=1)
# x = x/x.max()
# # # x = sep
# x = x.expand(1 ,3, 360, 640)
# # print(x.shape)

# import matplotlib.pyplot as plt
# # ds.ImgWrite("color.png", ds.TensorToImg(x))
# plt.imshow(ds.TensorToImg(x))
# plt.show()

In [ ]:
from dataset import TensorToImg, ImgWrite, ImgRead, ImgToTensor
from torch import from_numpy, Tensor, autograd, device
import MyRCNN
autograd.set_detect_anomaly(True)


model = MyRCNN.Model(device=device("cuda"))
model.train(dt, MyRCNN.MyBBLoss)
# x =  AvgPool2d(3, stride=3, padding=1)(x)
# x = ((x*255/32).round()*32).clamp(max=255)/255
# img = ds.TensorToImg(x)
# img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
# x = from_numpy(img).float().unsqueeze(0)/255
# N, M = x.shape[1:3]
# p = x.pow(2)
# sum = AvgPool2d(3, stride=1, padding=1)(x)*9
# sum2 = AvgPool2d(3, stride=1, padding=1)(p)*9
# x:Tensor = 9*p + sum2 - 2*x*sum
# x = 1 - x/x.max()
# x = (x*10).round()/10
# x[x<=0.9]=0
# x = x.unsqueeze(-1).expand(*x.shape,3).permute(0, -1, 1, 2)
# img = TensorToImg(x)
# ImgWrite("out.png", img)
# import matplotlib.pyplot as plt
# plt.imshow(img)
# plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch
x = dt.getTrainTensor(0).to(device=device("cuda"))
print(dt.getTrainLabel(0))
out: Tensor = model.model(x)
wh = out[:, 0:4, :, :]
score = torch.nn.functional.sigmoid(out[:, 0, :, :].unsqueeze(1))
H = out[:, 1, :, :].unsqueeze(1)
H = out[:, 2, :, :].unsqueeze(1)
h, w = score.shape[-2:]
row = torch.arange(h, device=score.device).view(1,1,h,1).expand(1,1,h,w)
col = torch.arange(w, device=score.device).view(1,1,1,w).expand(1,1,h,w)
cls = out[:, 3:, :, :].max(dim=1, keepdim=True).indices
print(score.max())
indices = (score > 0.79)
H = H[indices]
H = H[indices]
row = row[indices]
col = col[indices]
cls = cls[indices]
box = torch.stack([col, row, H, H], dim=-1).to(torch.device("cpu"))
print(cls.max())
for mat in box:
    X, Y, H, H = mat.detach().numpy()
    rect = patches.Rectangle([X - H, Y - H], H, H, edgecolor='red', facecolor='none')
    plt.subplot().add_patch(rect)
    
plt.imshow(TensorToImg(x.to(torch.device("cpu"))))
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch
x = dt.getTrainTensor(0).to(device=device("cuda"))
print(dt.getTrainLabel(0))
score: Tensor = model.model(x)
plt.imshow(TensorToImg(x.to(torch.device("cpu"))))
plt.show()
# color = model.model.color(mask, x)
# feature = model.model.feat(mask, color, x)
# x = color
# B, C, W, H = x.shape
# mask2 = model.model.color.ft(mask)
# x = feature[0, 4].unsqueeze(0).unsqueeze(0)
# x = x.permute(0, 2, 3, 1).view(x.shape[0], -1, 3)
# x = model.model.color.net1(x*mask * 2)
# x = model.model.color.net2(x*mask * 2)
# x = model.model.color.net3(x*mask * 2)
# x = model.model.color.merge(x)
# M: Tensor = x.view(x.shape[0], -1).max(dim=-1).values
# x = x.view(B, C, W, H) - x.min()
# x = color

# x -= x.min()
# x = x/x.max()
# x = x/x.max()
# x = x.expand(1, 3, -1, -1)
# x = x.unsqueeze(0).expand(1, 3, -1, -1)
# x = x - (1-mask)*10
# x = torch.nn.AvgPool2d(20, 20)(x)
# print(x.shape)
# ls: list[Tensor] = model.model(x.to(torch.device("cuda")))
# ls = [x.to(torch.device("cpu")) for x in ls]
# print(x.max())
# img = ds.TensorToImg(x.to(device("cpu")).detach())
# plt.imshow(img)
# print(MyRCNN.MyLoss(ls, dt.getTrainLabel(0).unsqueeze(0).to(x.device)))
# side: float = 1
# for mat in ls:
#     mat = torch.softmax(mat, dim=-1)
#     side *= 5
#     M, I = mat.max(dim=-1)
#     indices = (M > 0.6) & (I>0)
#     H, W = mat.shape[1:3]
#     row = torch.arange(H, device=x.device).view(1, H, 1).expand(1, H, W)
#     col = torch.arange(W, device=x.device).view(1, 1, W).expand(1, H, W)
#     # print(mat.shape)
#     label = mat[:, :, :, 4]
#     print(I[indices], M[indices])
#     position = torch.stack([row, col], dim=-1)[indices].to(device=device("cpu"))
#     for i in position:
#         rect = patches.Rectangle([i[1]*side,i[0]*side], side, side, edgecolor='red', facecolor='none')
#         plt.subplot().add_patch(rect)
# plt.show()s
